# Demo interativa — Classificador de Dígitos MNIST

Este notebook carrega o modelo MLP treinado no projeto e permite enviar uma imagem de um dígito manuscrito.

Fluxo:
1. Upload da imagem;
2. Pré-processamento no padrão MNIST;
3. Predição com a MLP;
4. Exibição da imagem processada;
5. Gráfico de probabilidades para as classes de 0 a 9.

## Pré-requisito

Antes de executar este notebook, salve o melhor modelo MLP treinado no notebook principal:

```python
from pathlib import Path
import joblib

Path("../models").mkdir(exist_ok=True)

joblib.dump(
    melhor_mlp,
    "../models/mlp_mnist.joblib"
)
```

Este notebook considera que está dentro da pasta `notebooks/` e que o modelo está em `models/mlp_mnist.joblib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import joblib
import ipywidgets as widgets

from pathlib import Path
from IPython.display import display, clear_output

In [ ]:
caminho_modelo = Path("../models/mlp_mnist.joblib")

if not caminho_modelo.exists():
    raise FileNotFoundError(
        "Modelo não encontrado. Salve primeiro o arquivo "
        "'../models/mlp_mnist.joblib' a partir do notebook principal."
    )

modelo = joblib.load(caminho_modelo)

print("Modelo carregado com sucesso.")
print("Classes:", modelo.classes_)

## Função de pré-processamento

A imagem enviada é convertida para escala de cinza, invertida quando necessário, recortada pela região do dígito, redimensionada preservando a proporção, centralizada em uma imagem 28 × 28 e normalizada para o intervalo [0, 1].

In [ ]:
def preparar_imagem(conteudo):
    array_bytes = np.frombuffer(conteudo, dtype=np.uint8)

    imagem = cv2.imdecode(
        array_bytes,
        cv2.IMREAD_GRAYSCALE
    )

    if imagem is None:
        raise ValueError("Não foi possível carregar a imagem.")

    if imagem.mean() > 127:
        imagem = 255 - imagem

    _, binaria = cv2.threshold(
        imagem,
        30,
        255,
        cv2.THRESH_BINARY
    )

    coordenadas = cv2.findNonZero(binaria)

    if coordenadas is None:
        raise ValueError("Nenhum dígito foi detectado na imagem.")

    x, y, w, h = cv2.boundingRect(coordenadas)

    recorte = imagem[
        y:y + h,
        x:x + w
    ]

    altura, largura = recorte.shape

    if altura > largura:
        nova_altura = 20
        nova_largura = max(
            1,
            int(round(largura * 20 / altura))
        )
    else:
        nova_largura = 20
        nova_altura = max(
            1,
            int(round(altura * 20 / largura))
        )

    redimensionada = cv2.resize(
        recorte,
        (nova_largura, nova_altura),
        interpolation=cv2.INTER_AREA
    )

    canvas = np.zeros(
        (28, 28),
        dtype=np.uint8
    )

    inicio_y = (28 - nova_altura) // 2
    inicio_x = (28 - nova_largura) // 2

    canvas[
        inicio_y:inicio_y + nova_altura,
        inicio_x:inicio_x + nova_largura
    ] = redimensionada

    momentos = cv2.moments(canvas)

    if momentos["m00"] != 0:
        centro_x = momentos["m10"] / momentos["m00"]
        centro_y = momentos["m01"] / momentos["m00"]

        deslocamento_x = int(round(13.5 - centro_x))
        deslocamento_y = int(round(13.5 - centro_y))

        matriz = np.float32([
            [1, 0, deslocamento_x],
            [0, 1, deslocamento_y]
        ])

        canvas = cv2.warpAffine(
            canvas,
            matriz,
            (28, 28),
            borderValue=0
        )

    imagem_modelo = (
        canvas.astype(np.float32) / 255.0
    ).reshape(1, 784)

    return canvas, imagem_modelo

## Upload e classificação

Selecione uma imagem PNG, JPG ou JPEG contendo um único dígito manuscrito.

In [ ]:
upload = widgets.FileUpload(
    accept=".png,.jpg,.jpeg",
    multiple=False,
    description="Enviar imagem"
)

saida = widgets.Output()

display(upload, saida)

In [ ]:
def obter_arquivo_upload(valor):
    if isinstance(valor, (tuple, list)):
        if len(valor) == 0:
            return None, None

        arquivo = valor[0]
        return arquivo["name"], bytes(arquivo["content"])

    if isinstance(valor, dict):
        if len(valor) == 0:
            return None, None

        nome = next(iter(valor))
        arquivo = valor[nome]
        conteudo = arquivo.get("content", arquivo)

        return nome, bytes(conteudo)

    return None, None


def classificar_upload(change):
    if not upload.value:
        return

    with saida:
        clear_output(wait=True)

        try:
            nome, conteudo = obter_arquivo_upload(
                upload.value
            )

            imagem_processada, entrada_modelo = preparar_imagem(
                conteudo
            )

            classe_prevista = modelo.predict(
                entrada_modelo
            )[0]

            probabilidades = modelo.predict_proba(
                entrada_modelo
            )[0]

            probabilidade_maxima = probabilidades.max()

            print(f"Arquivo: {nome}")
            print(f"Classe prevista: {classe_prevista}")
            print(f"Probabilidade: {probabilidade_maxima:.2%}")

            fig, axes = plt.subplots(
                1,
                2,
                figsize=(12, 4)
            )

            axes[0].imshow(
                imagem_processada,
                cmap="gray"
            )
            axes[0].set_title(
                f"Classe prevista: {classe_prevista}"
            )
            axes[0].axis("off")

            classes = modelo.classes_

            cores = [
                "tab:orange"
                if classe == classe_prevista
                else "tab:blue"
                for classe in classes
            ]

            axes[1].bar(
                classes,
                probabilidades,
                color=cores
            )

            axes[1].set_xticks(classes)
            axes[1].set_ylim(0, 1)
            axes[1].set_xlabel("Classe")
            axes[1].set_ylabel("Probabilidade")
            axes[1].set_title(
                "Probabilidade por classe"
            )

            plt.tight_layout()
            plt.show()

        except Exception as erro:
            print(f"Erro ao processar a imagem: {erro}")


upload.observe(
    classificar_upload,
    names="value"
)

## Interpretação

A barra destacada representa a classe selecionada pelo modelo. A probabilidade apresentada corresponde à maior probabilidade estimada pela MLP entre as classes conhecidas de 0 a 9.

Uma probabilidade elevada representa a confiança relativa do modelo entre as classes disponíveis e não garante, isoladamente, que a imagem esteja dentro da mesma distribuição do conjunto MNIST.